In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset , DataLoader
import pickle

In [3]:
data = datasets.MNIST('./Data',train = True , download = True)

In [ ]:
X = data.data
y = data.targets

X_train , X_test , y_train , y_test = train_test_split(X,y,test_size = 0.2,shuffle = True)
X_train =  X_train

X_train = X_train.float()/255

Y_train = y_train.long()

x_test = X_test.float()
x_test = x_test/255

y_test = y_test.long() 



In [5]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

device


device(type='cpu')

In [6]:
class Network(nn.Module):
  def __init__(self,in_channel):
    super().__init__()

    self.convo = nn.Sequential(
      nn.Conv2d(in_channel,32,3,padding = 'same'),
      nn.ReLU(),
      nn.BatchNorm2d(32),
      nn.MaxPool2d(2,2),       #32,14,14

      nn.Conv2d(32,64,3,padding = 'same'),
      nn.ReLU(),
      nn.BatchNorm2d(64),
      nn.MaxPool2d(2,2)     #64,7,7
    )

    self.linear = nn.Sequential(
        nn.Flatten(),
        nn.Linear(64*7*7,100),
        nn.ReLU(),
        nn.Dropout(p = 0.4),

        nn.Linear(100,75),
        nn.ReLU(),
        nn.Dropout(p = 0.3),

        nn.Linear(75,10)
    )

  def forward(self,X):
    X = self.convo(X)
    X = self.linear(X)
    return X



class Loader(Dataset):
  def __init__(self,x,y):
    self.x = torch.tensor(x,dtype = torch.float32).reshape(-1,1,28,28)
    self.y = torch.tensor(y,dtype = torch.long)
  def __len__(self):
    return len(self.x)
  def __getitem__(self,item):
    return self.x[item],self.y[item]


In [7]:
Train_loader = Loader(X_train,y_train)
Train_loader = DataLoader(Train_loader,batch_size = 32,shuffle = True)


test_loader = Loader(x_test,y_test)
test_loader = DataLoader(test_loader,batch_size = 32,shuffle = False)

C:\Users\sarma\AppData\Local\Temp\ipykernel_22992\1492513665.py:39: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.x = torch.tensor(x,dtype = torch.float32).reshape(-1,1,28,28)
C:\Users\sarma\AppData\Local\Temp\ipykernel_22992\1492513665.py:40: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.y = torch.tensor(y,dtype = torch.long)


In [8]:

model = Network(1).to(device)
optimizer = optim.Adam(model.parameters(),lr = 0.001)
criteria = nn.CrossEntropyLoss()

In [9]:
epoch = 20
for e in range(epoch):

  total_loss = 0

  for X,y in Train_loader:

    X = X.to(device)
    y = y.to(device)

    y_pred = model(X)

    loss = criteria(y_pred,y)

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    total_loss += loss.item()
  print(f"epoch = {e+1} , loss = {total_loss/len(Train_loader)}")



epoch = 1 , loss = 0.2130157802909768
epoch = 2 , loss = 0.0861000948336441
epoch = 3 , loss = 0.07282132223719964
epoch = 4 , loss = 0.05750286457111603
epoch = 5 , loss = 0.0507393313993501
epoch = 6 , loss = 0.044884252026976657
epoch = 7 , loss = 0.04442716718381462
epoch = 8 , loss = 0.04119980130898436
epoch = 9 , loss = 0.03534399511135325
epoch = 10 , loss = 0.03505210487226661
epoch = 11 , loss = 0.03423827310846748
epoch = 12 , loss = 0.027414040808471592
epoch = 13 , loss = 0.03103013065390446
epoch = 14 , loss = 0.02722172233922363
epoch = 15 , loss = 0.026055486990871134
epoch = 16 , loss = 0.02391203678271862
epoch = 17 , loss = 0.024619884907078088
epoch = 18 , loss = 0.022931218533351665
epoch = 19 , loss = 0.021824718113481998
epoch = 20 , loss = 0.020568058280945697


In [10]:
count = 0
total = 0
with torch.no_grad():
  for item , label in test_loader:

    item = item.to(device)

    label = label.to(device)

    y_out = model(item)

    predictions = torch.argmax(y_out, dim=1)
    total += label.shape[0]

    count += (predictions == label).sum().item()

print("correct --> ", count, " accuracy -->", (count / total) * 100, "%")

correct -->  11830  accuracy --> 98.58333333333333 %


In [11]:
print(torch.argmax(y_out,dim=1))
print(y_test)


tensor([0, 4, 0, 9, 0, 7, 4, 7, 2, 9, 6, 4, 1, 7, 4, 8, 2, 0, 2, 8, 8, 6, 7, 2,
        9, 7, 4, 4, 6, 6, 6, 0])
tensor([6, 9, 8,  ..., 6, 6, 0])


In [12]:
pickle.dump(model,open('modelv2.pkl','wb'))

In [57]:
from torchvision import transforms
from PIL import Image

input = Image.open("to_predict/image.png").convert("L")
transform = transforms.ToTensor()
input = transform(input)
print(input.shape)

input = input.reshape(1,1,28,28)
output = model(input)

result = torch.argmax(output,dim = 1)

print(result)

torch.Size([1, 28, 28])
tensor([3])
